# 05b ML Fixed Embeddings — Group 2: Fine-Tuned (Log Target)

**Group 2.** Trains regressors on fine-tuned GNN embeddings using a reconstruction loss objective and temporal warm-starting. Target is `log_systemic_risk_label`.

| Dataset | Model | Dim | Notes |
|---|---|---|---|
| `graphsage_v2_32_srisk_dataset.parquet` | GraphSAGE v2 | 32 | Reconstruction loss; 32-dim bottleneck |
| `graphsage_v2_64_srisk_dataset.parquet` | GraphSAGE v2 | 64 | Reconstruction loss; 64-dim bottleneck |
| `graphsage_v2_128_srisk_dataset.parquet` | GraphSAGE v2 | 128 | Reconstruction loss; 128-dim |
| `node2vec_v2_32_srisk_dataset.parquet` | Node2Vec v2 | 32 | Structural random walks; 32-dim |
| `node2vec_v2_64_srisk_dataset.parquet` | Node2Vec v2 | 64 | Structural random walks; 64-dim |
| `node2vec_v2_128_srisk_dataset.parquet` | Node2Vec v2 | 128 | Structural random walks; 128-dim |

> Run `03_g2_ref.ipynb` first to generate the parquet files.

In [1]:
from pathlib import Path
import sys
import os

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'requirements.txt').exists():
            return path
    raise FileNotFoundError('Project root not found.')


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from src.models.ml_train_and_store import (
    ModelTrainer,
    load_gnn_dataset,
    make_pipeline,
)

pd.set_option('display.max_columns', 200)
PROJECT_ROOT = find_project_root()
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis


## Load Datasets

In [2]:
DISPLAY_COLS = ["model", "train_mae", "validation_mae", "train_rmse", "validation_rmse"]
TOP1_COLS    = ["model", "train_top1_mae", "validation_top1_mae", "train_top1_rmse", "validation_top1_rmse"]

df_sage_32, feature_cols_sage_32 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="graphsage_v2_32_srisk_dataset.parquet",
)
print(f"GraphSAGE v2 32:  {df_sage_32.shape}  —  {len(feature_cols_sage_32)} embedding cols")

df_sage_64, feature_cols_sage_64 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="graphsage_v2_64_srisk_dataset.parquet",
)
print(f"GraphSAGE v2 64:  {df_sage_64.shape}  —  {len(feature_cols_sage_64)} embedding cols")

df_sage_128, feature_cols_sage_128 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="graphsage_v2_128_srisk_dataset.parquet",
)
print(f"GraphSAGE v2 128: {df_sage_128.shape}  —  {len(feature_cols_sage_128)} embedding cols")

df_n2v_32, feature_cols_n2v_32 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="node2vec_v2_32_srisk_dataset.parquet",
)
print(f"Node2Vec v2 32:   {df_n2v_32.shape}  —  {len(feature_cols_n2v_32)} embedding cols")

df_n2v_64, feature_cols_n2v_64 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="node2vec_v2_64_srisk_dataset.parquet",
)
print(f"Node2Vec v2 64:   {df_n2v_64.shape}  —  {len(feature_cols_n2v_64)} embedding cols")

df_n2v_128, feature_cols_n2v_128 = load_gnn_dataset(
    PROJECT_ROOT, target_col="log_systemic_risk_label",
    filename="node2vec_v2_128_srisk_dataset.parquet",
)
print(f"Node2Vec v2 128:  {df_n2v_128.shape}  —  {len(feature_cols_n2v_128)} embedding cols")

GraphSAGE v2 32:  (145536, 37)  —  32 embedding cols
GraphSAGE v2 64:  (145536, 69)  —  64 embedding cols
GraphSAGE v2 128: (145536, 133)  —  128 embedding cols
Node2Vec v2 32:   (145536, 37)  —  32 embedding cols
Node2Vec v2 64:   (145536, 69)  —  64 embedding cols
Node2Vec v2 128:  (145536, 133)  —  128 embedding cols


In [3]:
trainer_sage_32  = ModelTrainer(df=df_sage_32,  feature_cols=feature_cols_sage_32,  target_col="log_systemic_risk_label")
trainer_sage_64  = ModelTrainer(df=df_sage_64,  feature_cols=feature_cols_sage_64,  target_col="log_systemic_risk_label")
trainer_sage_128 = ModelTrainer(df=df_sage_128, feature_cols=feature_cols_sage_128, target_col="log_systemic_risk_label")
trainer_n2v_32   = ModelTrainer(df=df_n2v_32,   feature_cols=feature_cols_n2v_32,   target_col="log_systemic_risk_label")
trainer_n2v_64   = ModelTrainer(df=df_n2v_64,   feature_cols=feature_cols_n2v_64,   target_col="log_systemic_risk_label")
trainer_n2v_128  = ModelTrainer(df=df_n2v_128,  feature_cols=feature_cols_n2v_128,  target_col="log_systemic_risk_label")

print("GraphSAGE v2 32  —", trainer_sage_32.train_df.shape,  trainer_sage_32.val_df.shape)
print("GraphSAGE v2 64  —", trainer_sage_64.train_df.shape,  trainer_sage_64.val_df.shape)
print("GraphSAGE v2 128 —", trainer_sage_128.train_df.shape, trainer_sage_128.val_df.shape)
print("Node2Vec v2 32   —", trainer_n2v_32.train_df.shape,   trainer_n2v_32.val_df.shape)
print("Node2Vec v2 64   —", trainer_n2v_64.train_df.shape,   trainer_n2v_64.val_df.shape)
print("Node2Vec v2 128  —", trainer_n2v_128.train_df.shape,  trainer_n2v_128.val_df.shape)

GraphSAGE v2 32  — (109152, 37) (18192, 37)
GraphSAGE v2 64  — (109152, 69) (18192, 69)
GraphSAGE v2 128 — (109152, 133) (18192, 133)
Node2Vec v2 32   — (109152, 37) (18192, 37)
Node2Vec v2 64   — (109152, 69) (18192, 69)
Node2Vec v2 128  — (109152, 133) (18192, 133)


## Define Models

In [4]:
candidate_models = {
    "Linear Regression": make_pipeline(LinearRegression()),
    "Ridge":             make_pipeline(Ridge(alpha=1.0)),
    "MLP":               make_pipeline(MLPRegressor(hidden_layer_sizes=(100,), max_iter=500, random_state=42)),
    "Random Forest":     make_pipeline(RandomForestRegressor(n_estimators=100, random_state=42)),
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, random_state=42)),
    "XGBoost":           make_pipeline(XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42)),
}

list(candidate_models)

['Linear Regression',
 'Ridge',
 'MLP',
 'Random Forest',
 'Gradient Boosting',
 'XGBoost']

## Train — GraphSAGE v2 (32-dim)

In [5]:
trainer_sage_32.train_all(candidate_models)
display(trainer_sage_32.leaderboard()[DISPLAY_COLS])
trainer_sage_32.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Linear Regression,0.045,0.093,0.172,0.264
1,XGBoost,0.013,0.191,0.056,0.412
2,Gradient Boosting,0.011,0.265,0.053,0.567
3,Random Forest,0.007,0.305,0.036,0.695
4,Ridge,0.049,0.257,0.160,0.740
5,MLP,0.057,0.701,0.122,2.310


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Linear Regression,1.327,1.896,1.549,2.061
1,XGBoost,0.258,0.800,0.340,0.934
2,Gradient Boosting,0.233,0.790,0.329,0.932
3,Random Forest,0.205,0.921,0.262,1.071
4,Ridge,1.170,1.430,1.354,1.567
5,MLP,0.498,0.964,0.635,1.422


## Train — GraphSAGE v2 (64-dim)

In [6]:
trainer_sage_64.train_all(candidate_models)
display(trainer_sage_64.leaderboard()[DISPLAY_COLS])
trainer_sage_64.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Linear Regression,0.045,0.070,0.171,0.243
1,Gradient Boosting,0.009,0.101,0.047,0.287
2,XGBoost,0.010,0.113,0.049,0.292
3,Random Forest,0.006,0.145,0.034,0.329
4,MLP,0.055,0.214,0.137,0.551
5,Ridge,0.050,0.298,0.140,1.053


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Linear Regression,1.309,1.898,1.538,2.063
1,Gradient Boosting,0.192,1.218,0.284,1.363
2,XGBoost,0.213,1.236,0.285,1.368
3,Random Forest,0.189,1.154,0.242,1.280
4,MLP,0.431,1.385,0.555,1.538
5,Ridge,0.896,1.381,1.102,1.529


## Train — GraphSAGE v2 (128-dim)

In [7]:
trainer_sage_128.train_all(candidate_models)
display(trainer_sage_128.leaderboard()[DISPLAY_COLS])
trainer_sage_128.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,MLP,0.015,0.091,0.067,0.323
1,Gradient Boosting,0.009,0.155,0.048,0.348
2,XGBoost,0.010,0.177,0.047,0.405
3,Random Forest,0.006,0.222,0.035,0.511
4,Linear Regression,0.049,0.256,0.146,0.652
5,Ridge,0.045,0.533,0.125,1.736


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,MLP,0.344,1.081,0.450,1.275
1,Gradient Boosting,0.193,0.975,0.290,1.146
2,XGBoost,0.196,0.993,0.260,1.154
3,Random Forest,0.193,1.145,0.249,1.323
4,Linear Regression,1.002,1.632,1.211,1.785
5,Ridge,0.765,1.109,0.936,1.274


## Train — Node2Vec v2 (32-dim)

In [8]:
trainer_n2v_32.train_all(candidate_models)
display(trainer_n2v_32.leaderboard()[DISPLAY_COLS])
trainer_n2v_32.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.007,0.032,0.033,0.129
1,Gradient Boosting,0.013,0.030,0.061,0.133
2,XGBoost,0.013,0.031,0.061,0.136
3,MLP,0.022,0.038,0.075,0.137
4,Linear Regression,0.053,0.075,0.129,0.188
5,Ridge,0.053,0.075,0.129,0.188


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.156,0.633,0.207,0.745
1,Gradient Boosting,0.234,0.738,0.327,0.847
2,XGBoost,0.213,0.749,0.309,0.853
3,MLP,0.356,0.793,0.455,0.908
4,Linear Regression,0.780,1.336,0.942,1.406
5,Ridge,0.780,1.336,0.943,1.406


## Train — Node2Vec v2 (64-dim)

In [9]:
trainer_n2v_64.train_all(candidate_models)
display(trainer_n2v_64.leaderboard()[DISPLAY_COLS])
trainer_n2v_64.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.007,0.029,0.031,0.125
1,MLP,0.023,0.042,0.068,0.141
2,XGBoost,0.012,0.030,0.055,0.141
3,Gradient Boosting,0.012,0.031,0.057,0.147
4,Linear Regression,0.053,0.083,0.127,0.188
5,Ridge,0.053,0.083,0.127,0.188


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.147,0.641,0.191,0.748
1,MLP,0.298,0.826,0.380,0.954
2,XGBoost,0.168,0.916,0.243,1.015
3,Gradient Boosting,0.198,0.959,0.274,1.090
4,Linear Regression,0.761,1.269,0.918,1.365
5,Ridge,0.761,1.269,0.918,1.365


## Train — Node2Vec v2 (128-dim)

In [10]:
trainer_n2v_128.train_all(candidate_models)
display(trainer_n2v_128.leaderboard()[DISPLAY_COLS])
trainer_n2v_128.leaderboard()[TOP1_COLS]

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.006,0.034,0.032,0.129
1,Gradient Boosting,0.010,0.033,0.050,0.147
2,MLP,0.020,0.043,0.056,0.147
3,XGBoost,0.011,0.042,0.051,0.156
4,Ridge,0.044,0.062,0.117,0.176
5,Linear Regression,0.044,0.062,0.117,0.176


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.154,0.606,0.202,0.721
1,Gradient Boosting,0.151,0.891,0.245,1.020
2,MLP,0.218,0.908,0.310,1.025
3,XGBoost,0.156,0.883,0.231,1.003
4,Ridge,0.741,1.337,0.895,1.399
5,Linear Regression,0.741,1.337,0.895,1.399


## Top 1% Leaderboards

In [11]:
print("GraphSAGE v2 (32-dim)");  display(trainer_sage_32.leaderboard()[TOP1_COLS])
print("GraphSAGE v2 (64-dim)");  display(trainer_sage_64.leaderboard()[TOP1_COLS])
print("GraphSAGE v2 (128-dim)"); display(trainer_sage_128.leaderboard()[TOP1_COLS])
print("Node2Vec v2 (32-dim)");   display(trainer_n2v_32.leaderboard()[TOP1_COLS])
print("Node2Vec v2 (64-dim)");   display(trainer_n2v_64.leaderboard()[TOP1_COLS])
print("Node2Vec v2 (128-dim)");  display(trainer_n2v_128.leaderboard()[TOP1_COLS])

GraphSAGE v2 (32-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Linear Regression,1.327,1.896,1.549,2.061
1,XGBoost,0.258,0.800,0.340,0.934
2,Gradient Boosting,0.233,0.790,0.329,0.932
3,Random Forest,0.205,0.921,0.262,1.071
4,Ridge,1.170,1.430,1.354,1.567
5,MLP,0.498,0.964,0.635,1.422


GraphSAGE v2 (64-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Linear Regression,1.309,1.898,1.538,2.063
1,Gradient Boosting,0.192,1.218,0.284,1.363
2,XGBoost,0.213,1.236,0.285,1.368
3,Random Forest,0.189,1.154,0.242,1.280
4,MLP,0.431,1.385,0.555,1.538
5,Ridge,0.896,1.381,1.102,1.529


GraphSAGE v2 (128-dim)


...

,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.156,0.633,0.207,0.745
1,Gradient Boosting,0.234,0.738,0.327,0.847
2,XGBoost,0.213,0.749,0.309,0.853
3,MLP,0.356,0.793,0.455,0.908
4,Linear Regression,0.780,1.336,0.942,1.406
5,Ridge,0.780,1.336,0.943,1.406


Node2Vec v2 (64-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.147,0.641,0.191,0.748
1,MLP,0.298,0.826,0.380,0.954
2,XGBoost,0.168,0.916,0.243,1.015
3,Gradient Boosting,0.198,0.959,0.274,1.090
4,Linear Regression,0.761,1.269,0.918,1.365
5,Ridge,0.761,1.269,0.918,1.365


Node2Vec v2 (128-dim)


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.154,0.606,0.202,0.721
1,Gradient Boosting,0.151,0.891,0.245,1.020
2,MLP,0.218,0.908,0.310,1.025
3,XGBoost,0.156,0.883,0.231,1.003
4,Ridge,0.741,1.337,0.895,1.399
5,Linear Regression,0.741,1.337,0.895,1.399


## Hyperparameter Tuning

In [12]:
def tune(trainer, base_model, param_distributions, name, n_iter=40):
    X = pd.concat([trainer.train_df[trainer.feature_cols], trainer.val_df[trainer.feature_cols]])
    y = pd.concat([trainer.train_df[trainer.target_col],   trainer.val_df[trainer.target_col]])
    split_idx = np.concatenate([
        np.full(len(trainer.train_df), -1),
        np.zeros(len(trainer.val_df), dtype=int),
    ])
    search = RandomizedSearchCV(
        base_model, param_distributions,
        n_iter=n_iter, cv=PredefinedSplit(split_idx),
        scoring="neg_root_mean_squared_error",
        random_state=42, n_jobs=-1,
    )
    search.fit(X, y)
    trainer.train(search.best_estimator_, name=name)
    trainer.best_params[name] = search.best_params_
    return search.best_params_


RF_PARAMS = {
    "model__n_estimators":      [100, 200, 300],
    "model__max_depth":         [None, 5, 10],
    "model__min_samples_leaf":  [1, 2, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10, 15, 20],
    "model__max_features":      ["sqrt", "log2", 0.5, 0.8, 1.0],
}
GB_PARAMS = {
    "model__max_iter":          [100, 200, 300],
    "model__max_depth":         [3, 5, 8, None],
    "model__learning_rate":     [0.005, 0.01, 0.05],
    "model__min_samples_leaf":  [5, 10, 20, 50, 100],
    "model__l2_regularization": [1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__max_leaf_nodes":    [15, 20, 30, 40, 50, 60],
    "model__max_bins":          [64, 128, 255],
}
XGB_PARAMS = {
    "model__n_estimators":     [100, 200, 400],
    "model__max_depth":        [3, 4, 5, 6, 8, 10],
    "model__learning_rate":    [0.005, 0.01, 0.05],
    "model__subsample":        [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.5, 0.6, 0.7, 0.8, 1.0],
    "model__min_child_weight": [1, 2, 5, 10],
    "model__gamma":            [0, 0.1, 0.5, 1.0, 2.0],
    "model__reg_alpha":        [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
    "model__reg_lambda":       [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0],
}
MLP_PARAMS = {
    "model__hidden_layer_sizes": [(64,), (128,), (128, 64), (256, 128), (128, 64, 32), (256, 128, 64)],
    "model__activation":         ["relu", "tanh"],
    "model__alpha":              [1e-5, 1e-4, 1e-3, 1e-2, 0.1],
    "model__learning_rate_init": [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 0.05],
    "model__learning_rate":      ["constant", "adaptive"],
    "model__batch_size":         [32, 64, 128, "auto"],
}

for label, t in [
    ("GraphSAGE v2 (32-dim)",  trainer_sage_32),
    ("GraphSAGE v2 (64-dim)",  trainer_sage_64),
    ("GraphSAGE v2 (128-dim)", trainer_sage_128),
    ("Node2Vec v2 (32-dim)",   trainer_n2v_32),
    ("Node2Vec v2 (64-dim)",   trainer_n2v_64),
    ("Node2Vec v2 (128-dim)",  trainer_n2v_128),
]:
    print(f"\n===== Tuning {label} =====")
    tune(t, make_pipeline(RandomForestRegressor(random_state=42)),         RF_PARAMS,  "Random Forest (tuned)")
    tune(t, make_pipeline(HistGradientBoostingRegressor(random_state=42)), GB_PARAMS,  "Gradient Boosting (tuned)")
    tune(t, make_pipeline(XGBRegressor(random_state=42)),                  XGB_PARAMS, "XGBoost (tuned)")
    tune(t, make_pipeline(MLPRegressor(max_iter=500, early_stopping=True, n_iter_no_change=5, random_state=42)), MLP_PARAMS, "MLP (tuned)")
    display(t.leaderboard()[DISPLAY_COLS])
    display(t.leaderboard()[TOP1_COLS])


===== Tuning GraphSAGE v2 (32-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,MLP (tuned),0.020,0.050,0.058,0.142
1,Gradient Boosting (tuned),0.035,0.080,0.153,0.228
2,XGBoost (tuned),0.034,0.077,0.147,0.232
3,Linear Regression,0.045,0.093,0.172,0.264
4,Random Forest (tuned),0.030,0.167,0.135,0.389
5,XGBoost,0.013,0.191,0.056,0.412
6,Gradient Boosting,0.011,0.265,0.053,0.567
7,Random Forest,0.007,0.305,0.036,0.695
8,Ridge,0.049,0.257,0.160,0.740
9,MLP,0.057,0.701,0.122,2.310


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,MLP (tuned),0.268,0.710,0.357,0.884
1,Gradient Boosting (tuned),1.177,1.745,1.396,1.899
2,XGBoost (tuned),1.139,1.710,1.326,1.848
3,Linear Regression,1.327,1.896,1.549,2.061
4,Random Forest (tuned),1.011,1.438,1.193,1.581
5,XGBoost,0.258,0.800,0.340,0.934
6,Gradient Boosting,0.233,0.790,0.329,0.932
7,Random Forest,0.205,0.921,0.262,1.071
8,Ridge,1.170,1.430,1.354,1.567
9,MLP,0.498,0.964,0.635,1.422



===== Tuning GraphSAGE v2 (64-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,MLP (tuned),0.025,0.041,0.081,0.168
1,XGBoost (tuned),0.031,0.050,0.139,0.214
2,Gradient Boosting (tuned),0.032,0.051,0.139,0.214
3,Random Forest (tuned),0.022,0.068,0.108,0.224
4,Linear Regression,0.045,0.070,0.171,0.243
5,Gradient Boosting,0.009,0.101,0.047,0.287
6,XGBoost,0.010,0.113,0.049,0.292
7,Random Forest,0.006,0.145,0.034,0.329
8,MLP,0.055,0.214,0.137,0.551
9,Ridge,0.050,0.298,0.140,1.053


...

,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest (tuned),0.008,0.028,0.040,0.125
1,Random Forest,0.007,0.029,0.031,0.125
2,MLP (tuned),0.017,0.032,0.070,0.132
3,MLP,0.023,0.042,0.068,0.141
4,XGBoost,0.012,0.030,0.055,0.141
5,XGBoost (tuned),0.014,0.031,0.065,0.143
6,Gradient Boosting (tuned),0.017,0.032,0.080,0.144
7,Gradient Boosting,0.012,0.031,0.057,0.147
8,Linear Regression,0.053,0.083,0.127,0.188
9,Ridge,0.053,0.083,0.127,0.188


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest (tuned),0.169,0.661,0.230,0.763
1,Random Forest,0.147,0.641,0.191,0.748
2,MLP (tuned),0.302,0.674,0.393,0.809
3,MLP,0.298,0.826,0.380,0.954
4,XGBoost,0.168,0.916,0.243,1.015
5,XGBoost (tuned),0.249,0.950,0.345,1.045
6,Gradient Boosting (tuned),0.403,0.927,0.505,1.012
7,Gradient Boosting,0.198,0.959,0.274,1.090
8,Linear Regression,0.761,1.269,0.918,1.365
9,Ridge,0.761,1.269,0.918,1.365



===== Tuning Node2Vec v2 (128-dim) =====


,model,train_mae,validation_mae,train_rmse,validation_rmse
0,Random Forest,0.006,0.034,0.032,0.129
1,Random Forest (tuned),0.008,0.034,0.039,0.129
2,MLP (tuned),0.016,0.038,0.059,0.140
3,Gradient Boosting,0.010,0.033,0.050,0.147
4,MLP,0.020,0.043,0.056,0.147
5,XGBoost (tuned),0.011,0.037,0.050,0.149
6,Gradient Boosting (tuned),0.017,0.034,0.080,0.149
7,XGBoost,0.011,0.042,0.051,0.156
8,Ridge,0.044,0.062,0.117,0.176
9,Linear Regression,0.044,0.062,0.117,0.176


,model,train_top1_mae,validation_top1_mae,train_top1_rmse,validation_top1_rmse
0,Random Forest,0.154,0.606,0.202,0.721
1,Random Forest (tuned),0.175,0.636,0.235,0.741
2,MLP (tuned),0.219,0.834,0.311,0.950
3,Gradient Boosting,0.151,0.891,0.245,1.020
4,MLP,0.218,0.908,0.310,1.025
5,XGBoost (tuned),0.162,0.918,0.224,1.019
6,Gradient Boosting (tuned),0.416,0.929,0.518,1.039
7,XGBoost,0.156,0.883,0.231,1.003
8,Ridge,0.741,1.337,0.895,1.399
9,Linear Regression,0.741,1.337,0.895,1.399


## Comparison with Baseline Results

Reference results from `04_ML_Classic_Algorithms.ipynb` and `05_ML_GNN_Embeddings.ipynb`:

| Dataset | Best model | Train MAE | Val MAE | Train RMSE | Val RMSE |
|---|---|---|---|---|---|
| Classical features (DebtRank, PageRank, ...) | XGBoost (tuned) | 0.0068 | 0.0139 | 0.0409 | 0.0824 |
| GraphSAGE v1 (link prediction, 64-dim) | XGBoost (tuned) | 0.0110 | 0.0351 | 0.0639 | 0.2275 |
| Node2Vec v1 (64-dim) | XGBoost (tuned) | 0.0130 | 0.0367 | 0.0623 | 0.2050 |

Fixed embeddings (this notebook) — best model per dataset:

In [13]:
def best_row(trainer, label):
    row = trainer.leaderboard().iloc[0]
    return {
        "Dataset": label,
        "Best model": row["model"],
        "Train MAE": round(float(row["train_mae"]), 4),
        "Val MAE":   round(float(row["validation_mae"]), 4),
        "Train RMSE": round(float(row["train_rmse"]), 4),
        "Val RMSE":   round(float(row["validation_rmse"]), 4),
    }

comparison = pd.DataFrame([
    best_row(trainer_sage_32,  "GraphSAGE v2 (reconstruction, 32-dim)"),
    best_row(trainer_sage_64,  "GraphSAGE v2 (reconstruction, 64-dim)"),
    best_row(trainer_sage_128, "GraphSAGE v2 (reconstruction, 128-dim)"),
    best_row(trainer_n2v_32,   "Node2Vec v2 (structural, 32-dim)"),
    best_row(trainer_n2v_64,   "Node2Vec v2 (structural, 64-dim)"),
    best_row(trainer_n2v_128,  "Node2Vec v2 (structural, 128-dim)"),
])

comparison.set_index("Dataset")

,Best model,Train MAE,Val MAE,Train RMSE,Val RMSE
Dataset,,,,,
"GraphSAGE v2 (reconstruction, 32-dim)",MLP (tuned),0.020,0.050,0.058,0.142
"GraphSAGE v2 (reconstruction, 64-dim)",MLP (tuned),0.025,0.041,0.081,0.168
"GraphSAGE v2 (reconstruction, 128-dim)",MLP (tuned),0.028,0.051,0.118,0.176
"Node2Vec v2 (structural, 32-dim)",Random Forest (tuned),0.012,0.031,0.057,0.127
"Node2Vec v2 (structural, 64-dim)",Random Forest (tuned),0.008,0.028,0.040,0.125
"Node2Vec v2 (structural, 128-dim)",Random Forest,0.006,0.034,0.032,0.129


## Optional — Inspect Best Hyperparameters / Save Models

In [14]:
# Best hyperparameters found during tuning
all_trainers = [
    ("GraphSAGE v2 (32-dim)",  trainer_sage_32),
    ("GraphSAGE v2 (64-dim)",  trainer_sage_64),
    ("GraphSAGE v2 (128-dim)", trainer_sage_128),
    ("Node2Vec v2 (32-dim)",   trainer_n2v_32),
    ("Node2Vec v2 (64-dim)",   trainer_n2v_64),
    ("Node2Vec v2 (128-dim)",  trainer_n2v_128),
]
for label, t in all_trainers:
    if t.best_params:
        print(f"\n{'='*50}\n{label}")
        for model_name, params in t.best_params.items():
            print(f"  {model_name}:")
            for k, v in params.items():
                print(f"    {k.replace('model__', '')}: {v}")


GraphSAGE v2 (32-dim)
  Random Forest (tuned):
    n_estimators: 300
    min_samples_split: 15
    min_samples_leaf: 20
    max_features: sqrt
    max_depth: 5
  Gradient Boosting (tuned):
    min_samples_leaf: 100
    max_leaf_nodes: 40
    max_iter: 100
    max_depth: 5
    max_bins: 128
    learning_rate: 0.005
    l2_regularization: 0.01
  XGBoost (tuned):
    subsample: 1.0
    reg_lambda: 1.0
    reg_alpha: 0.0001
    n_estimators: 100
    min_child_weight: 2
    max_depth: 5
    learning_rate: 0.005
    gamma: 1.0
    colsample_bytree: 1.0
  MLP (tuned):
    learning_rate_init: 0.0005
    learning_rate: adaptive
    hidden_layer_sizes: (256, 128, 64)
    batch_size: auto
    alpha: 0.001
    activation: tanh

GraphSAGE v2 (64-dim)
  Random Forest (tuned):
    n_estimators: 300
    min_samples_split: 20
    min_samples_leaf: 10
    max_features: 1.0
    max_depth: 5
  Gradient Boosting (tuned):
    min_samples_leaf: 10
    max_leaf_nodes: 30
    max_iter: 100
    max_depth: 3
  

In [15]:
# Save the best models to disk (edit the list below)
from src.models.ml_train_and_store import load_model

SAVE_DIR = PROJECT_ROOT / "src" / "models" / "dataset_1" / "05_b"

for label, t in all_trainers:
    best_name = t.leaderboard().iloc[0]["model"]
    path = t.save_model(best_name, SAVE_DIR)

# To reload later:
# model = load_model(path)

Saved 'MLP (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\dataset_1\05_b\MLP_(tuned).joblib
Saved 'MLP (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\dataset_1\05_b\MLP_(tuned).joblib
Saved 'MLP (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\dataset_1\05_b\MLP_(tuned).joblib
Saved 'Random Forest (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\dataset_1\05_b\Random_Forest_(tuned).joblib
Saved 'Random Forest (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\dataset_1\05_b\Random_Forest_(tuned).joblib
Saved 'Random Forest' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\dataset_1\05_b\Random_Forest.joblib


In [16]:
# Save n2v 32 with Gradient Boosting tunned
# Save n2v 32 with Gradient Boosting tuned
SAVE_DIR = PROJECT_ROOT / "src" / "models" / "dataset_1" / "05_b"
path_n2v_32_gb = trainer_n2v_32.save_model("Gradient Boosting (tuned)", SAVE_DIR)

# To reload later:
# from src.models.ml_train_and_store import load_model
# model = load_model(path_n2v_32_gb)

Saved 'Gradient Boosting (tuned)' → C:\Users\ruben\Desktop\Universidade\Nova IMS\Tese\Thesis\src\models\dataset_1\05_b\Gradient_Boosting_(tuned).joblib
